# Extra Person Contextual Risk — with Head Pose & Gaze

## Design Philosophy

> A second person visible in the frame is **not automatically cheating**.
> We use **positional proximity signals** (YOLO bounding boxes),
> **head pose** (yaw/pitch/roll from MediaPipe Face Landmarker transformation matrix),
> and **gaze direction** (iris landmark offsets) to build a richer contextual risk signal.
> We never claim cheating from one image alone.

### Risk Flag Naming
| Flag | Meaning |
|------|---------|
| `EXTRA_PERSON_PRESENT_LOW` | Background / far / brief / candidate faces camera |
| `EXTRA_PERSON_CONTEXTUAL_RISK_MEDIUM` | Moderate proximity or candidate looks sideways |
| `POSSIBLE_HUMAN_ASSISTANCE_HIGH` | Close + candidate yaw/gaze toward extra person |

### Pipeline
```
YOLO11m (person only)
    → identify candidate (largest + most central)
    → identify extra persons
    → assess_single_image_risk()   [position + proximity]
        ↓
MediaPipe Face Landmarker
    → head pose  (yaw / pitch / roll)
    → gaze       (iris offset → gaze_x)
    → is_looking_toward_extra()    [engagement signal]
        ↓
Combined risk escalation
    → final flag + reason
```
---

## 1. Imports

In [ ]:
import sys
import cv2
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
from IPython.display import display

# Add src/ to path
sys.path.insert(0, str(Path("src").resolve()))

from extra_person_risk import (
    identify_main_candidate,
    assess_single_image_risk,
    draw_extra_person_risk,
    TemporalExtraPersonTracker,
)

from head_pose_gaze import (
    ensure_landmarker_model,
    create_face_landmarker,
    analyse_head_pose_and_gaze,
    is_looking_toward_extra,
    draw_head_pose_gaze,
)

print("All imports OK")

## 2. Config

In [ ]:
IMAGE_DIR    = Path("images")
OUT_DIR      = Path("outputs/extra_person_risk")
OUT_DIR.mkdir(parents=True, exist_ok=True)

YOLO_MODEL  = "yolo12x.pt"   # YOLO11 medium — better accuracy than YOLOv8s
PERSON_CONF = 0.35
PERSON_CLS  = [0]            # COCO class 0 = person only

# Head pose: yaw threshold (degrees) to flag candidate as looking toward extra person
# Lower = more sensitive, Higher = fewer false positives
YAW_LOOK_THRESHOLD = 20.0

image_paths = sorted(IMAGE_DIR.glob("*.png")) + sorted(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(image_paths)} images")

## 3. Load Models

YOLO11m will download on first run (~40 MB).  
Face Landmarker will download on first run (~29 MB).

In [ ]:
# YOLO
yolo = YOLO(YOLO_MODEL)
print(f"✓ Loaded {YOLO_MODEL}")

# MediaPipe Face Landmarker
ensure_landmarker_model()
landmarker = create_face_landmarker(num_faces=2)
print("✓ Face Landmarker ready")

## 4. Run Full Pipeline on All Images

In [ ]:
all_results = []

for img_path in image_paths:
    image_bgr = cv2.imread(str(img_path))
    if image_bgr is None:
        continue
    h, w = image_bgr.shape[:2]

    # ── Step 1: YOLO person detection ─────────────────────────────────────
    yolo_out = yolo(str(img_path), conf=PERSON_CONF, classes=PERSON_CLS,
                    imgsz=640, verbose=False)
    person_boxes = []
    for res in yolo_out:
        for box in res.boxes:
            if int(box.cls[0]) == 0:
                person_boxes.append({
                    "bbox":       [round(v, 1) for v in box.xyxy[0].tolist()],
                    "confidence": round(float(box.conf[0]), 3),
                })

    candidate  = identify_main_candidate(person_boxes, w, h)
    extra_list = [b for b in person_boxes if b is not candidate]

    # ── Step 2: Positional risk (YOLO-only) ───────────────────────────────
    if candidate:
        base_risk = assess_single_image_risk(candidate, extra_list, w, h)
    else:
        base_risk = {
            "extra_person_present": False, "extra_person_count": 0,
            "overall_risk_level": "NONE", "overall_risk_flag": "NO_PERSON_DETECTED",
            "overall_reason": "No person detected.", "extra_person_details": [],
            "limitation": None,
        }

    # ── Step 3: Head pose + gaze ───────────────────────────────────────────
    face_results = analyse_head_pose_and_gaze(image_bgr, landmarker)

    # Pick the candidate's face (face_index=0 = largest/first detected face)
    candidate_face = face_results[0] if face_results else None

    # ── Step 4: Engagement signal ──────────────────────────────────────────
    engagement_signals = []
    looking_toward_any = False

    if candidate_face and base_risk.get("extra_person_details"):
        for extra_detail in base_risk["extra_person_details"]:
            signal = is_looking_toward_extra(
                candidate_pose=candidate_face,
                extra_person_horizontal_zone=extra_detail["position"],
            )
            engagement_signals.append(signal)
            if signal["looking_toward"]:
                looking_toward_any = True

    # ── Step 5: Combined risk escalation ──────────────────────────────────
    # If the base positional risk is already HIGH, keep it.
    # If MEDIUM + candidate is looking toward extra → escalate to HIGH.
    # If LOW + candidate is looking toward extra → escalate to MEDIUM.
    base_level = base_risk["overall_risk_level"]
    final_level = base_level
    final_flag  = base_risk.get("overall_risk_flag", "")
    final_reason = base_risk.get("overall_reason", "")

    head_pose_note = ""
    if candidate_face:
        pose = candidate_face["head_pose"]
        gaze = candidate_face["gaze"]
        head_pose_note = (
            f"Head pose: yaw={pose['raw_yaw']:.0f}° ({pose['horizontal']}), "
            f"pitch={pose['raw_pitch']:.0f}° ({pose['vertical']}). "
            f"Gaze: {gaze['gaze_label']} (x={gaze['gaze_x']:.2f})."
        )

    if looking_toward_any and base_level == "MEDIUM":
        final_level  = "HIGH"
        final_flag   = "POSSIBLE_HUMAN_ASSISTANCE_HIGH"
        final_reason = (
            base_risk.get("overall_reason", "") + " " +
            head_pose_note + " Candidate head/gaze directed toward extra person — escalated to HIGH."
        )
    elif looking_toward_any and base_level == "LOW":
        final_level  = "MEDIUM"
        final_flag   = "EXTRA_PERSON_CONTEXTUAL_RISK_MEDIUM"
        final_reason = (
            base_risk.get("overall_reason", "") + " " +
            head_pose_note + " Candidate appears to look toward extra person — escalated to MEDIUM."
        )
    elif candidate_face:
        final_reason = base_risk.get("overall_reason", "") + " " + head_pose_note

    # ── Step 6: Save annotated image ──────────────────────────────────────
    annotated = image_bgr.copy()
    annotated = draw_extra_person_risk(annotated, base_risk)
    annotated = draw_head_pose_gaze(annotated, face_results)
    save_path = OUT_DIR / f"{img_path.stem}_full_risk.jpg"
    cv2.imwrite(str(save_path), annotated)

    # ── Log ───────────────────────────────────────────────────────────────
    pose_str = (
        f"yaw={candidate_face['head_pose']['raw_yaw']:.0f}° "
        f"gaze={candidate_face['gaze']['gaze_label']}"
        if candidate_face else "no_face"
    )
    print(f"{img_path.name:20s}  persons={len(person_boxes)}  "
          f"risk={final_level:6s}  {pose_str}")

    all_results.append({
        "image":             img_path.name,
        "total_persons":     len(person_boxes),
        "extra_count":       len(extra_list),
        "base_risk_level":   base_level,
        "final_risk_level":  final_level,
        "final_risk_flag":   final_flag,
        "final_reason":      final_reason,
        "face_detected":     candidate_face is not None,
        "head_yaw":          candidate_face["head_pose"]["raw_yaw"] if candidate_face else None,
        "head_pitch":        candidate_face["head_pose"]["raw_pitch"] if candidate_face else None,
        "head_direction":    candidate_face["head_pose"]["horizontal"] if candidate_face else None,
        "gaze_x":            candidate_face["gaze"]["gaze_x"] if candidate_face else None,
        "gaze_label":        candidate_face["gaze"]["gaze_label"] if candidate_face else None,
        "looking_toward_extra": looking_toward_any,
        "annotated_path":    str(save_path),
    })

print(f"\nDone. Processed {len(all_results)} images.")

## 5. Save Reports

In [ ]:
df = pd.DataFrame(all_results)

json_path = Path("outputs/reports/full_risk_report.json")
with open(json_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"JSON → {json_path}")

csv_path = Path("outputs/reports/full_risk_report.csv")
df.drop(columns=["final_reason", "annotated_path"]).to_csv(csv_path, index=False)
print(f"CSV  → {csv_path}")

pd.set_option("display.max_colwidth", 30)
display(df[[
    "image", "total_persons", "extra_count",
    "base_risk_level", "final_risk_level",
    "head_direction", "gaze_label", "looking_toward_extra"
]])

## 6. Summary Statistics

In [ ]:
from collections import Counter

total = len(df)
print(f"Total images : {total}")
print(f"\n── Final Risk Levels ──")
for lvl, cnt in Counter(df["final_risk_level"]).most_common():
    print(f"  {lvl:<8} : {cnt}")

print(f"\n── Head Pose ──")
print(f"  Face detected           : {df['face_detected'].sum()} / {total}")
for d, cnt in Counter(df["head_direction"].dropna()).most_common():
    print(f"  {d:<25} : {cnt}")

print(f"\n── Gaze ──")
for g, cnt in Counter(df["gaze_label"].dropna()).most_common():
    print(f"  {g:<25} : {cnt}")

print(f"\n── Engagement signal ──")
print(f"  looking_toward_extra=True  : {df['looking_toward_extra'].sum()}")
print(f"  Risk escalated by head pose: "
      f"{(df['final_risk_level'] != df['base_risk_level']).sum()}")

# ── Charts ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Full Risk Pipeline — YOLO + Head Pose + Gaze", fontsize=13, fontweight="bold")

color_map = {"NONE": "#9E9E9E", "LOW": "#FDD835", "MEDIUM": "#FB8C00", "HIGH": "#E53935"}

# 1. Base vs Final risk
ax = axes[0]
x       = range(len(df))
base_colors  = [color_map.get(v, "gray") for v in df["base_risk_level"]]
final_colors = [color_map.get(v, "gray") for v in df["final_risk_level"]]
ax.scatter(x, df["base_risk_level"].map({"NONE":0,"LOW":1,"MEDIUM":2,"HIGH":3}),
           c=base_colors, marker="o", label="Base", alpha=0.7)
ax.scatter(x, df["final_risk_level"].map({"NONE":0,"LOW":1,"MEDIUM":2,"HIGH":3}),
           c=final_colors, marker="^", label="Final", alpha=0.7)
ax.set_yticks([0,1,2,3])
ax.set_yticklabels(["NONE","LOW","MEDIUM","HIGH"])
ax.set_title("Base vs Final Risk per Image")
ax.set_xlabel("Image index")
ax.legend()

# 2. Head direction distribution
ax = axes[1]
hd_counts = df["head_direction"].value_counts()
hd_colors = ["#4CAF50" if "camera" in k else "#FF5722" for k in hd_counts.index]
ax.bar(hd_counts.index, hd_counts.values, color=hd_colors, edgecolor="white")
ax.set_title("Head Direction Distribution")
ax.set_ylabel("Images")
for i, v in enumerate(hd_counts.values):
    ax.text(i, v + 0.2, str(v), ha="center", fontsize=9)

# 3. Gaze label distribution
ax = axes[2]
gz_counts = df["gaze_label"].value_counts()
gz_colors = ["#4CAF50" if "center" in str(k) else "#FF9800" for k in gz_counts.index]
ax.bar(gz_counts.index, gz_counts.values, color=gz_colors, edgecolor="white")
ax.set_title("Gaze Direction Distribution")
ax.set_ylabel("Images")
for i, v in enumerate(gz_counts.values):
    ax.text(i, v + 0.2, str(v), ha="center", fontsize=9)

plt.tight_layout()
chart_path = Path("outputs/reports/full_risk_chart.png")
plt.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Chart saved → {chart_path}")

## 7. Spot-Check — Annotated Images with Pose + Risk Overlay

- **Green box** = main candidate
- **Yellow / Orange / Red box** = extra person (LOW / MEDIUM / HIGH)
- **Green arrow** = head pose direction (candidate's face)
- **Orange dot** = iris gaze position
- **Banner at bottom** = final risk flag

In [ ]:
risk_filter = None   # None = show all  |  "HIGH" / "MEDIUM" / "LOW"
SHOW_N      = 6

sample = df[df["final_risk_level"] == risk_filter] if risk_filter else df
sample = sample.head(SHOW_N)

files = [Path(r["annotated_path"]) for _, r in sample.iterrows()]

cols = 3
rows = max(1, (len(files) + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 5))
axes = axes.flatten()

for i, (f, (_, row)) in enumerate(zip(files, sample.iterrows())):
    img = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(
        f"{row['image']}\n"
        f"risk={row['final_risk_level']}  persons={row['total_persons']}\n"
        f"yaw={row['head_yaw']}°  gaze={row['gaze_label']}",
        fontsize=8
    )
    axes[i].axis("off")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Full Risk Pipeline — Annotated Samples", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Temporal Tracker — using cached results

Feeds all single-image results into the temporal tracker **without re-running YOLO**.
In a real session, call `tracker.add_frame()` per screenshot/video frame.

In [ ]:
from extra_person_risk import TemporalExtraPersonTracker

tracker = TemporalExtraPersonTracker(fps=1)

# Count frames where head pose pointed toward extra person
# (already computed in all_results — no re-running YOLO)
head_pose_toward_count = int(df["looking_toward_extra"].sum())

# Build minimal frame results from cached data for the tracker
for r in all_results:
    tracker.add_frame({
        "extra_person_present": r["extra_count"] > 0,
        "extra_person_count":   r["extra_count"],
        "extra_person_details": [
            # Minimal placeholder — tracker only needs normalised_distance
            {"normalised_distance": 0.20}  # approximation; wire real data for production
        ] if r["extra_count"] > 0 else [],
    })

temporal = tracker.analyze(
    head_pose_toward_extra_count = head_pose_toward_count,
    audio_activity_count         = 0,   # wire Silero VAD output here
    other_risk_flag_count        = 0,   # wire phone/book flags here
)

print("═" * 60)
print("TEMPORAL ANALYSIS RESULT")
print("═" * 60)
for k, v in temporal.items():
    print(f"  {k:<40} : {v}")

## 9. Accuracy Notes & Limitations

| Signal | Source | Accuracy | Notes |
|--------|--------|---------|-------|
| Person count | YOLO11m | Good | May over-count reflections/posters |
| Main candidate ID | Largest + most central box | Good for webcam | Fails if candidate moves to corner |
| Extra person proximity | Normalised bbox distance | Good | No depth — 2D approximation |
| Head yaw/pitch | FaceLandmarker 4x4 matrix | ±5–10° | Degrades with extreme angles |
| Gaze direction | Iris landmark offset | Coarse (left/right/center) | Not pixel-accurate gaze |
| Engagement signal | yaw + gaze combined | Medium | Temporal confirmation needed |

### How to improve gaze accuracy
- **MediaPipe Iris** (legacy solutions API) gives tighter iris tracking if available
- **OpenFace** gives AU-based gaze vectors — more accurate, requires separate build
- **Temporal smoothing**: average yaw + gaze_x over N frames before triggering a flag

### Next step: audio
Wire **Silero VAD** or **WebRTC VAD** output into `temporal.analyze(audio_activity_count=N)`
to add a voice activity signal that further refines HIGH risk classification.